# Capstone - Which pages should a content editor open first?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhalid04/Shaheer-Khalid-FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Deployed paper: https://shaheerkhalid04.github.io/Shaheer-Khalid-FlyRank-ML-Internship/**

Lane 2, Refresh / Content Opportunity Scoring. This notebook is the evidence behind the paper: it builds the panel, defines the label, trains the models, runs the leakage probes, and scores the sealed month once.

**The headline, up front, because it is a negative result and those are easy to bury:** on a sealed future month a gradient-boosted model was indistinguishable from a two-line transparent rule at ranking, and roughly 47 times worse at protecting traffic. The recommendation is to ship the rule.

In [1]:
import json
import os
import time

import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

SEED = 42
CUTOFF_TRAIN = ["2026-01", "2026-02", "2026-03"]
CUTOFF_VAL = "2026-04"
CUTOFF_TEST = "2026-05"          # its label month is June 2026, the sealed final month

FEATURES = [
    "log_imp_m0", "imp_90d", "momentum", "momentum_prior", "ctr", "position",
    "position_coverage", "position_delta", "ctr_deficit", "active_day_ratio", "clk_m0",
]
print(f"seed={SEED} | {len(FEATURES)} features | sealed test cutoff {CUTOFF_TEST} -> June 2026")

seed=42 | 11 features | sealed test cutoff 2026-05 -> June 2026


## 1. Question

**Which pages should a content editor open first this week?**

Not "which pages are declining". In the sealed test month about 65% of pages with real demand were losing ground, so a yes/no flag lights up two thirds of the library and answers nothing. The editor's constraint is time, not information, so the only useful output is an **order**.

That fixes the shape of the work:

- **Unit:** one page, observed at one cutoff date.
- **Output:** a ranked queue, each row carrying a score, one reason code and an action label.
- **Metric:** precision at a queue depth K that matches real review capacity, plus how much at-risk traffic that queue actually covers.
- **Wrong-call cost:** a false positive burns an editor's hour; a false negative lets a page with real demand bleed unwatched. Capacity is binding, so the expensive error is at the top of the list.

## 2. Data

The pseudonymised warehouse release, read over the network with DuckDB. Nothing was downloaded in bulk and nothing client-identifying appears anywhere: clients and pages are opaque hashes, and no domains, URLs or queries are shown.

The heavy step is a single scan of eight monthly partitions aggregated to a **page x month panel**, cached locally. Every cutoff below is derived from that cache, so the 58M-row scan happens once.

**Deliberately excluded, each one costly:**

| Excluded | Why |
|---|---|
| every date column in `dim_content` | `last_optimized_date` has 45,396 non-null values and the earliest is 2026-04-24, after every cutoff. It is a current-state dimension with no history |
| `fact_content_query_90d` | its fixed 90-day window overlaps the label period |
| product flags and existing scores | learning them teaches the old rule, not the world |

The `dim_content` exclusion killed my best signal from earlier weeks (staleness). I would rather lose a feature than train on the future.

In [2]:
# The panel. One warehouse scan, cached; everything downstream reads the cache.
PANEL = "work/outputs/capstone_page_month_panel.parquet"
MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]

if os.path.exists(PANEL):
    panel = pd.read_parquet(PANEL)
    print(f"loaded cached panel: {len(panel):,} page-months")
else:
    import duckdb
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    con = duckdb.connect()
    con.execute("SET enable_progress_bar=false")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')" if token
                else "CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
    REL = "hf://datasets/FlyRank/internship-warehouse"
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS)
    t = time.time()
    panel = con.sql(f'''
        SELECT client_hash_id, content_hash_id,
               STRFTIME(report_date, '%Y-%m')                                        AS ym,
               SUM(gsc_impressions)                                                  AS impressions,
               SUM(gsc_clicks)                                                       AS clicks,
               SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS pos_sum_valid,
               SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END)  AS imp_pos_valid,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)    AS active_days
        FROM read_parquet([{paths}])
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2, 3
    ''').df()
    os.makedirs("work/outputs", exist_ok=True)
    panel.to_parquet(PANEL, index=False)
    print(f"scanned and cached {len(panel):,} page-months in {time.time() - t:.0f}s")

print(panel.groupby("ym").agg(pages=("content_hash_id", "nunique"),
                              impressions=("impressions", "sum")).to_string())

loaded cached panel: 1,302,189 page-months


          pages  impressions
ym                          
2025-11   99403   83871948.0
2025-12  109639  112406941.0
2026-01  121544  144354689.0
2026-02  153559  180128922.0
2026-03  176738  280657589.0
2026-04  194760  292067218.0
2026-05  237910  266164899.0
2026-06  208636  216194872.0


## 3. Methodology

### The label, and why it is measured per day

```text
eligible : impressions in the cutoff month >= 50
label    : y = 1 if mean daily impressions NEXT month < 0.8 x mean daily impressions THIS month
features : cutoff month and the two before it, nothing after
```

Impressions are normalised **per day**. Calendar months differ in length, so an unnormalised month-over-month ratio partly measures February being three days short. The normalised and raw labels agree on 97.2% of rows, so the choice tightens the definition without manufacturing the result.

### Five time-ordered cutoffs

Train on Jan, Feb, Mar 2026. Validate on April. **June is sealed** and scored once at the very end.

### The frozen baseline

Two inputs, no fitted weights, carried unchanged from the baseline notebook:

```text
score = ctr_deficit_vs_position_band_peers  x  log(1 + monthly impressions)
```

The logarithm is not cosmetic. Raw demand is negatively associated with decline here, so multiplying by raw impressions turns the score into a volume ranking and makes it worse.

In [3]:
# Build the five cutoffs from the panel.
FRAME = "work/outputs/capstone_model_frame.parquet"
DAYS = {"2025-11": 30, "2025-12": 31, "2026-01": 31, "2026-02": 28,
        "2026-03": 31, "2026-04": 30, "2026-05": 31, "2026-06": 30}
CUTOFFS = [("2026-01", "2025-12", "2025-11", "2026-02"),
           ("2026-02", "2026-01", "2025-12", "2026-03"),
           ("2026-03", "2026-02", "2026-01", "2026-04"),
           ("2026-04", "2026-03", "2026-02", "2026-05"),
           ("2026-05", "2026-04", "2026-03", "2026-06")]

if os.path.exists(FRAME):
    frame = pd.read_parquet(FRAME)
    print(f"loaded cached modelling frame: {len(frame):,} page-cutoff rows")
else:
    panel["key"] = panel["client_hash_id"] + "|" + panel["content_hash_id"]
    by_month = {m: g.set_index("key") for m, g in panel.groupby("ym")}
    parts = []
    for m0, m1, m2, lm in CUTOFFS:
        cur = by_month[m0][["client_hash_id", "content_hash_id", "impressions", "clicks",
                            "pos_sum_valid", "imp_pos_valid", "active_days"]].rename(
            columns={"impressions": "imp_m0", "clicks": "clk_m0", "active_days": "active_days_m0"})
        prev = by_month[m1][["impressions", "pos_sum_valid", "imp_pos_valid"]].rename(
            columns={"impressions": "imp_m1"})
        df = cur.join(prev[["imp_m1"]], how="left")
        df = df.join(prev[["pos_sum_valid", "imp_pos_valid"]].add_suffix("_m1"), how="left")
        df = df.join(by_month[m2][["impressions"]].rename(columns={"impressions": "imp_m2"}), how="left")
        df = df.join(by_month[lm][["impressions"]].rename(columns={"impressions": "imp_next"}), how="left")
        for c in ("imp_m1", "imp_m2", "imp_next"):
            df[c] = df[c].fillna(0)
        df = df[df["imp_m0"] >= 50].copy()

        df["position"] = df["pos_sum_valid"] / df["imp_pos_valid"].replace(0, np.nan)
        df["position_coverage"] = df["imp_pos_valid"] / df["imp_m0"]
        df["position_delta"] = df["position"] - (df["pos_sum_valid_m1"] / df["imp_pos_valid_m1"].replace(0, np.nan))
        df["ctr"] = df["clk_m0"] / df["imp_m0"] * 100
        df["momentum"] = df["imp_m0"] / df["imp_m1"].replace(0, np.nan)
        df["momentum_prior"] = df["imp_m1"] / df["imp_m2"].replace(0, np.nan)
        df["imp_90d"] = df["imp_m0"] + df["imp_m1"] + df["imp_m2"]
        df["log_imp_m0"] = np.log1p(df["imp_m0"])
        df["active_day_ratio"] = df["active_days_m0"] / 31.0
        df["position_band"] = pd.cut(df["position"], [0, 3, 10, 20, 50, 1000],
                                     labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
        peer = df.groupby("position_band", observed=True).apply(
            lambda g: g["clk_m0"].sum() / g["imp_m0"].sum() * 100, include_groups=False)
        df["peer_ctr"] = df["position_band"].map(peer).astype(float)
        df["ctr_deficit"] = ((df["peer_ctr"] - df["ctr"]) / df["peer_ctr"]).clip(0, 1)

        d0, dn = DAYS[m0], DAYS[lm]
        df["imp_per_day_m0"] = df["imp_m0"] / d0
        df["imp_per_day_next"] = df["imp_next"] / dn
        df["y"] = (df["imp_per_day_next"] < 0.8 * df["imp_per_day_m0"]).astype(int)
        df["y_raw_monthly"] = (df["imp_next"] < 0.8 * df["imp_m0"]).astype(int)
        df["cutoff"], df["label_month"] = m0, lm
        parts.append(df.reset_index(drop=True))
    frame = pd.concat(parts, ignore_index=True)
    frame.to_parquet(FRAME, index=False)
    print(f"built {len(frame):,} page-cutoff rows")

# DuckDB does not promise row order; sort so the split is reproducible.
frame = frame.sort_values(["cutoff", "client_hash_id", "content_hash_id"]).reset_index(drop=True)
frame["baseline_score"] = (frame["ctr_deficit"] * np.log1p(frame["imp_m0"])).fillna(0.0)

print()
print(frame.groupby("cutoff").agg(rows=("y", "size"), clients=("client_hash_id", "nunique"),
                                  base_rate=("y", "mean"),
                                  base_rate_unnormalised=("y_raw_monthly", "mean")).round(4).to_string())
print()
print(f"per-day and raw labels agree on {(frame.y == frame.y_raw_monthly).mean() * 100:.1f}% of rows")

loaded cached modelling frame: 543,573 page-cutoff rows



           rows  clients  base_rate  base_rate_unnormalised
cutoff                                                     
2026-01   74328       30     0.2119                  0.2618
2026-02   93654       38     0.2764                  0.2293
2026-03  116114       44     0.4984                  0.5184
2026-04  125758       47     0.5504                  0.5310
2026-05  133719       53     0.6489                  0.6648



per-day and raw labels agree on 97.2% of rows


**The base rate is not stable, and that is a finding rather than a nuisance.** It runs 0.21, 0.28, 0.50, 0.55, 0.65 across the five cutoffs. The panel shifts from rapid growth to contraction over these months, so the thing being predicted genuinely changes. Two consequences I carry through the rest of the notebook: precision at K is never comparable across months without its base rate beside it, and a model trained on the early cutoffs is answering a slightly different question by the late ones.

In [4]:
# Split. Time-ordered, and the sealed month is untouched until the final evaluation.
train = frame[frame.cutoff.isin(CUTOFF_TRAIN)].reset_index(drop=True)
val = frame[frame.cutoff == CUTOFF_VAL].reset_index(drop=True)
test = frame[frame.cutoff == CUTOFF_TEST].reset_index(drop=True)

for nm, d in (("train", train), ("validation", val), ("SEALED test", test)):
    print(f"{nm:12s} n={len(d):>7,}  clients={d.client_hash_id.nunique():>3}  "
          f"base rate={d.y.mean():.4f}  cutoffs={sorted(d.cutoff.unique())}")

overlap = set(train.client_hash_id) & set(test.client_hash_id)
print(f"\nclients appearing in both train and sealed test: {len(overlap)}")
print("(expected: the split is by TIME here, so clients recur. the grouped CV below is what")
print(" tests generalisation to unseen clients.)")

train        n=284,096  clients= 48  base rate=0.3502  cutoffs=['2026-01', '2026-02', '2026-03']
validation   n=125,758  clients= 47  base rate=0.5504  cutoffs=['2026-04']
SEALED test  n=133,719  clients= 53  base rate=0.6489  cutoffs=['2026-05']



clients appearing in both train and sealed test: 44
(expected: the split is by TIME here, so clients recur. the grouped CV below is what
 tests generalisation to unseen clients.)


### Leakage checks

Four probes, run before believing anything. The first one is the important one: a harness that cannot detect leakage tells you nothing when it stays quiet.

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score


def p_at_k(df, score, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")
    return float(df.iloc[order[:k]]["y"].mean())


def impression_recall_at_k(df, score, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")
    top = df.iloc[order[:k]]
    return float(top.loc[top.y == 1, "imp_m0"].sum() / df.loc[df.y == 1, "imp_m0"].sum())


def gb():
    return HistGradientBoostingClassifier(random_state=SEED, max_iter=300,
                                          learning_rate=0.06, max_leaf_nodes=31)


# PROBE 1: deliberately leak the label window and confirm the harness screams.
leak_tr, leak_te = train.copy(), test.copy()
leak_tr["leak"] = leak_tr["imp_per_day_next"]
leak_te["leak"] = leak_te["imp_per_day_next"]
m_leak = gb().fit(leak_tr[FEATURES + ["leak"]], leak_tr.y)
auc_leaked = roc_auc_score(leak_te.y, m_leak.predict_proba(leak_te[FEATURES + ["leak"]])[:, 1])

m_honest = gb().fit(train[FEATURES], train.y)
probs_test = m_honest.predict_proba(test[FEATURES])[:, 1]
auc_honest = roc_auc_score(test.y, probs_test)

print(f"PROBE 1  with a label-derived feature : AUC {auc_leaked:.4f}")
print(f"         honest                       : AUC {auc_honest:.4f}")
print(f"         gap                          : {auc_leaked - auc_honest:+.4f}  <- harness works")

PROBE 1  with a label-derived feature : AUC 0.9997
         honest                       : AUC 0.5925
         gap                          : +0.4072  <- harness works


In [6]:
from sklearn.model_selection import GroupKFold, KFold

# PROBE 2: how much of any score is client memorisation?
oof_grouped = np.full(len(train), np.nan)
for tr_i, te_i in GroupKFold(n_splits=5).split(train, train.y, groups=train.client_hash_id):
    oof_grouped[te_i] = gb().fit(train.iloc[tr_i][FEATURES], train.iloc[tr_i].y) \
                            .predict_proba(train.iloc[te_i][FEATURES])[:, 1]

oof_random = np.full(len(train), np.nan)
for tr_i, te_i in KFold(n_splits=5, shuffle=True, random_state=SEED).split(train):
    oof_random[te_i] = gb().fit(train.iloc[tr_i][FEATURES], train.iloc[tr_i].y) \
                           .predict_proba(train.iloc[te_i][FEATURES])[:, 1]

auc_grouped = roc_auc_score(train.y, oof_grouped)
auc_random = roc_auc_score(train.y, oof_random)
print(f"PROBE 2  random 5-fold  AUC {auc_random:.4f}")
print(f"         grouped 5-fold AUC {auc_grouped:.4f}   (folds never share a client)")
print(f"         memorisation gap   {auc_random - auc_grouped:+.4f}")
print("         -> every number reported from here uses the grouped or time-based split.")

PROBE 2  random 5-fold  AUC 0.7202
         grouped 5-fold AUC 0.6488   (folds never share a client)
         memorisation gap   +0.0714
         -> every number reported from here uses the grouped or time-based split.


In [7]:
# PROBE 3: window audit. Every feature must come from the cutoff month or earlier.
windows = {
    "log_imp_m0": "cutoff month", "imp_90d": "cutoff month + 2 before",
    "momentum": "cutoff vs prior month", "momentum_prior": "two months before cutoff",
    "ctr": "cutoff month", "position": "cutoff month", "position_coverage": "cutoff month",
    "position_delta": "cutoff vs prior month", "ctr_deficit": "cutoff month only",
    "active_day_ratio": "cutoff month", "clk_m0": "cutoff month",
}
assert set(windows) == set(FEATURES)
print("PROBE 3  feature windows")
for f in FEATURES:
    print(f"         {f:20s} <- {windows[f]}")
print(f"         label                <- the month AFTER the cutoff, never a feature")

# PROBE 4: population audit. Eligibility must not consult the outcome window.
print()
print("PROBE 4  eligibility rule = 'impressions in the cutoff month >= 50'")
print("         uses no column from the label month. no survivorship filter applied.")

PROBE 3  feature windows
         log_imp_m0           <- cutoff month
         imp_90d              <- cutoff month + 2 before
         momentum             <- cutoff vs prior month
         momentum_prior       <- two months before cutoff
         ctr                  <- cutoff month
         position             <- cutoff month
         position_coverage    <- cutoff month
         position_delta       <- cutoff vs prior month
         ctr_deficit          <- cutoff month only
         active_day_ratio     <- cutoff month
         clk_m0               <- cutoff month
         label                <- the month AFTER the cutoff, never a feature

PROBE 4  eligibility rule = 'impressions in the cutoff month >= 50'
         uses no column from the label month. no survivorship filter applied.


## 4. Results, model against baseline on the same splits

Two results that disagree, which is the interesting part.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))


def row(df, score, name):
    return {"method": name, "AUC": round(roc_auc_score(df.y, score), 4),
            "AP": round(average_precision_score(df.y, score), 4),
            "P@50": round(p_at_k(df, score, 50), 3),
            "P@200": round(p_at_k(df, score, 200), 3),
            "imp_recall@200": round(impression_recall_at_k(df, score, 200), 4),
            "lift@200": round(p_at_k(df, score, 200) / df.y.mean(), 3)}


oof_lr = np.full(len(train), np.nan)
for tr_i, te_i in GroupKFold(n_splits=5).split(train, train.y, groups=train.client_hash_id):
    oof_lr[te_i] = lr().fit(train.iloc[tr_i][FEATURES], train.iloc[tr_i].y) \
                       .predict_proba(train.iloc[te_i][FEATURES])[:, 1]

grouped = pd.DataFrame([
    row(train, oof_grouped, "gradient boosting"),
    row(train, train.baseline_score, "transparent rule"),
    row(train, oof_lr, "logistic regression"),
])
print(f"GROUPED CROSS-VALIDATION, training cutoffs   (base rate {train.y.mean():.4f}, n={len(train):,})")
print(grouped.to_string(index=False))
print()
print("on this evidence alone I would have shipped the model.")

GROUPED CROSS-VALIDATION, training cutoffs   (base rate 0.3502, n=284,096)
             method    AUC     AP  P@50  P@200  imp_recall@200  lift@200
  gradient boosting 0.6488 0.4773  0.62  0.700          0.0019     1.999
   transparent rule 0.5782 0.4058  0.64  0.625          0.0305     1.784
logistic regression 0.5697 0.3900  0.46  0.535          0.0199     1.527

on this evidence alone I would have shipped the model.


In [9]:
# The sealed month. Scored once.
m_lr = lr().fit(train[FEATURES], train.y)
test = test.copy()
test["p_model"] = probs_test
test["p_lr"] = m_lr.predict_proba(test[FEATURES])[:, 1]
test["value_weighted"] = test["p_model"] * np.log1p(test["imp_m0"])

sealed = pd.DataFrame([
    row(test, test.baseline_score, "transparent rule"),
    row(test, test.p_model, "gradient boosting"),
    row(test, test.value_weighted, "model x log(demand)"),
    row(test, test.p_lr, "logistic regression"),
])
print(f"SEALED TEST, June 2026   (base rate {test.y.mean():.4f}, n={len(test):,})")
print(sealed.to_string(index=False))

print()
print("median page size in each top 200:")
for nm, s in (("transparent rule", test.baseline_score), ("gradient boosting", test.p_model),
              ("model x log(demand)", test.value_weighted)):
    idx = np.argsort(-np.asarray(s, float), kind="stable")[:200]
    print(f"  {nm:22s} {test.iloc[idx].imp_m0.median():>10,.0f} impressions")

SEALED TEST, June 2026   (base rate 0.6489, n=133,719)
             method    AUC     AP  P@50  P@200  imp_recall@200  lift@200
   transparent rule 0.5909 0.7105  0.78  0.830          0.0427     1.279
  gradient boosting 0.5925 0.7083  0.84  0.805          0.0009     1.241
model x log(demand) 0.5816 0.7048  0.78  0.825          0.0266     1.271
logistic regression 0.5945 0.6881  0.56  0.670          0.0570     1.033

median page size in each top 200:
  transparent rule           28,658 impressions
  gradient boosting             186 impressions
  model x log(demand)        14,942 impressions


**Read the last two columns, not the first.**

All three AUCs land between 0.591 and 0.595. On a sealed future month the model and a two-line rule are indistinguishable at ranking. The model wins the very top of the queue (P@50 of 0.840 against 0.780) and gives it back by K=200.

The decisive difference is `imp_recall@200`: the rule's top 200 covers **4.27%** of at-risk impressions, the model's covers **0.09%**. About 47 times less. The reason is in the median page size, 186 impressions against 28,658. The model optimised the metric it was given, and that metric counted pages as if they were equal.

Multiplying the model's probability by log demand recovers most of the value (2.66%) but then does no better than the rule it was meant to replace.

In [10]:
# What the model leans on, and where it is wrong.
from sklearn.inspection import permutation_importance

sample = test.sample(n=25000, random_state=SEED)
perm = permutation_importance(m_honest, sample[FEATURES], sample.y, n_repeats=5,
                              random_state=SEED, scoring="roc_auc", n_jobs=-1)
imp = pd.DataFrame({"feature": FEATURES, "AUC_drop_when_shuffled": perm.importances_mean,
                    "std": perm.importances_std}).sort_values("AUC_drop_when_shuffled",
                                                              ascending=False)
print("permutation importance on the sealed test:")
print(imp.round(4).to_string(index=False))
print()
print("the top feature is active_day_ratio, which the rule does not use at all.")
print("average position contributes nothing measurable - shuffling it did not hurt.")

permutation importance on the sealed test:
          feature  AUC_drop_when_shuffled    std
 active_day_ratio                  0.0489 0.0043
              ctr                  0.0316 0.0019
         momentum                  0.0275 0.0026
   position_delta                  0.0088 0.0020
           clk_m0                  0.0077 0.0006
position_coverage                  0.0060 0.0004
      ctr_deficit                  0.0046 0.0007
       log_imp_m0                  0.0039 0.0001
          imp_90d                  0.0011 0.0009
   momentum_prior                 -0.0003 0.0014
         position                 -0.0036 0.0027

the top feature is active_day_ratio, which the rule does not use at all.
average position contributes nothing measurable - shuffling it did not hurt.


In [11]:
# Can the model's discovery be folded back into the rule? Chosen on TRAIN only.
band = pd.cut(train.active_day_ratio, [0, .5, .8, .95, 1.01],
              labels=["under half", "50-80%", "80-95%", "95-100%"])
tbl = train.groupby(band, observed=True).agg(n=("y", "size"), decline_rate=("y", "mean"))
print(f"train months, base rate {train.y.mean():.4f}")
print(tbl.round(3).to_string())
print()
print("20.3% -> 44.8% -> 30.2% -> 39.1%: not monotone, so there is no threshold to write.")
print("the attempt to turn the model's best feature into a readable rule FAILED, and the")
print("frozen rule stands unchanged. a negative result, reported as one.")

train months, base rate 0.3502
                       n  decline_rate
active_day_ratio                      
under half         24928         0.203
50-80%             38040         0.448
80-95%            101496         0.302
95-100%           119632         0.391

20.3% -> 44.8% -> 30.2% -> 39.1%: not monotone, so there is no threshold to write.
the attempt to turn the model's best feature into a readable rule FAILED, and the
frozen rule stands unchanged. a negative result, reported as one.


In [12]:
# Calibration under drift, and three confident mistakes.
test["decile"] = pd.qcut(test["p_model"], 10, labels=False, duplicates="drop")
cal = test.groupby("decile").agg(n=("y", "size"), predicted=("p_model", "mean"),
                                 observed=("y", "mean"))
print("calibration on the sealed month (trained where 35% declined, tested where 65% did):")
print(cal.round(3).to_string())
print()
wrong = test[(test.p_model > 0.8) & (test.y == 0)]
print(f"confident and wrong: {len(wrong):,} rows. the three biggest:")
print(wrong.nlargest(3, "imp_m0")[["imp_m0", "clk_m0", "position", "momentum", "p_model"]]
      .round(2).to_string(index=False))
print()
print("the first one grew 108x month-over-month then held: a spike the model read as fragility.")
print("ranking survives this miscalibration; the probabilities should not be shown to anyone.")

calibration on the sealed month (trained where 35% declined, tested where 65% did):
            n  predicted  observed
decile                            
0       13372      0.146     0.402
1       13372      0.254     0.608
2       13372      0.315     0.644
3       13372      0.363     0.662
4       13372      0.404     0.664
5       13371      0.440     0.679
6       13372      0.474     0.688
7       13372      0.509     0.697
8       13401      0.550     0.712
9       13343      0.639     0.733

confident and wrong: 117 rows. the three biggest:
  imp_m0  clk_m0  position  momentum  p_model
159173.0   238.0      5.77    108.80     0.81
 21607.0    13.0      6.85      0.22     0.88
 15785.0   277.0      6.39     15.86     0.87

the first one grew 108x month-over-month then held: a spike the model read as fragility.
ranking survives this miscalibration; the probabilities should not be shown to anyone.


## 5. Limitations

What this work cannot say, written before a reader has to ask.

- **Observational, not causal.** Nothing here shows that refreshing a page recovers traffic. No intervention was made. The output ranks pages for attention; it promises no outcome.
- **Not a statement about search engines.** These are outcomes measured in one pseudonymised portfolio over eight months. Any pattern may belong to this portfolio, this period, or how these sites are built.
- **Selective panel.** Of 104 clients, 24 have six or more months of history at my main cutoff and 37 have no recorded start date. This describes well-instrumented clients, not a site three weeks into onboarding.
- **Unstable base rate.** 0.21 to 0.65 across five cutoffs, so precision at K is not comparable across months and lift is reported beside it everywhere.
- **A 30-day horizon cannot separate a page going bad from a topic going quiet.** Seasonality is not modelled.
- **Absence is not zero.** The daily table accrues only from a page's registration day, so early gaps mean "not watched", not "earned nothing". This biases 90-day demand downward for newer pages.
- **One sealed month is one observation.** Scored once, which is the right discipline, but it cannot tell me whether the rule's advantage is durable or a property of June's contraction.

The strongest claim the evidence carries: *on GSC-instrumented clients between November 2025 and June 2026, a two-input transparent rule ranked pages for editorial review about as accurately as a gradient-boosted model, and substantially better by protected traffic.*

## 6. Ranked recommendations

1. **Ship the rule, not the model.** Same ranking quality on unseen future data, 47x more protected traffic, readable in one line, no training pipeline, cannot drift silently. Where two options perform the same, ship the one a person can argue with.
2. **Change the objective before training anything else.** The model was not broken, it was pointed at the wrong target. Select future models on impression-weighted recall at K so that being right about an invisible page counts for what it is worth.
3. **Surface position coverage in the queue.** Pages whose impressions mostly carry no ranking position decline at 67.8% against 47.1%. They belong in the queue, but the reason code should say the position data is thin, not that the title underperforms.
4. **Split off the six-figure-impression, single-digit-click pages.** They dominate the top of the rule's queue and are rarely a title problem. They need a query-mix diagnosis, not a rewrite.
5. **Recalibrate monthly and treat the base rate as a health metric.** The panel moved from 21% to 65% decline in five months. No per-page queue will ever surface that.

In [13]:
# The queue an editor would actually receive, from the sealed month.
queue = test.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

big_deficit = queue["ctr_deficit"] >= 0.5
high_demand = queue["imp_m0"] >= 1000
queue["action"] = np.select([big_deficit & high_demand, big_deficit, queue["ctr_deficit"] > 0],
                            ["REFRESH", "REVIEW", "MONITOR"], default="PROTECT")
queue["reason_code"] = np.select(
    [big_deficit & high_demand, big_deficit, queue["ctr_deficit"] > 0],
    ["ctr_far_below_peers_high_demand", "ctr_far_below_peers", "ctr_slightly_below_peers"],
    default="ctr_at_or_above_peers")

cols = ["rank", "content_hash_id", "imp_m0", "clk_m0", "position", "position_coverage",
        "ctr_deficit", "action", "reason_code"]
queue[cols].to_csv("work/outputs/capstone_action_queue.csv", index=False)
print(f"wrote work/outputs/capstone_action_queue.csv  ({len(queue):,} rows, no label column)")
print()
print(queue.head(10)[["rank", "imp_m0", "clk_m0", "position", "position_coverage",
                      "action", "reason_code"]].round(3).to_string(index=False))

wrote work/outputs/capstone_action_queue.csv  (133,719 rows, no label column)

 rank   imp_m0  clk_m0  position  position_coverage  action                     reason_code
    1 189298.0     2.0    10.080              1.000 REFRESH ctr_far_below_peers_high_demand
    2 219982.0    52.0     2.563              0.667 REFRESH ctr_far_below_peers_high_demand
    3 235817.0    45.0     5.781              1.000 REFRESH ctr_far_below_peers_high_demand
    4 191566.0    32.0     8.697              1.000 REFRESH ctr_far_below_peers_high_demand
    5 112353.0     4.0     3.377              0.321 REFRESH ctr_far_below_peers_high_demand
    6 109052.0     1.0    84.424              1.000 REFRESH ctr_far_below_peers_high_demand
    7  88175.0     1.0     8.261              1.000 REFRESH ctr_far_below_peers_high_demand
    8  82667.0     0.0    10.070              1.000 REFRESH ctr_far_below_peers_high_demand
    9 118966.0    16.0     7.771              1.000 REFRESH ctr_far_below_peers_high_demand
 

## 7. Artifacts the paper embeds

The deployed paper shows four charts, all regenerated here into `docs/img/` so the page and the notebook can never disagree.

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("docs/img", exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.facecolor": "white"})
INK, ACCENT, MUTED = "#1c2530", "#c2410c", "#94a3b8"
BASE = test.y.mean()

ks = [20, 50, 100, 200, 500, 1000]
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(ks, [p_at_k(test, test.baseline_score, k) for k in ks], "o-", color=ACCENT, lw=2,
        label="transparent rule")
ax.plot(ks, [p_at_k(test, test.p_model, k) for k in ks], "s-", color=INK, lw=2,
        label="gradient boosting")
ax.axhline(BASE, ls="--", color=MUTED, lw=1.5, label=f"base rate ({BASE:.2f})")
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("queue depth K (pages an editor reviews)"); ax.set_ylabel("precision@K")
ax.set_title("Sealed June test: the model does not separate from the rule", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9); ax.set_ylim(0.5, 1.0)
fig.tight_layout(); fig.savefig("docs/img/precision_at_k.svg"); plt.close(fig)

variants = [("baseline rule", test.baseline_score), ("model probability", test.p_model),
            ("model x log(demand)", test.value_weighted)]
vals = [impression_recall_at_k(test, s, 200) * 100 for _, s in variants]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
bars = ax.barh([n for n, _ in variants], vals, color=[ACCENT, INK, "#0f766e"], height=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_width() + 0.15, b.get_y() + b.get_height() / 2, f"{v:.2f}%", va="center", fontsize=9.5)
ax.set_xlabel("share of at-risk impressions captured in the top 200")
ax.set_title("Same precision, wildly different business value", loc="left", fontsize=11)
ax.set_xlim(0, max(vals) * 1.25)
fig.tight_layout(); fig.savefig("docs/img/impression_recall.svg"); plt.close(fig)

PRETTY = {"log_imp_m0": "demand (log impressions)", "imp_90d": "90-day demand",
          "momentum": "momentum (this month / last)", "momentum_prior": "prior momentum",
          "ctr": "click-through rate", "position": "average position",
          "position_coverage": "position coverage", "position_delta": "position movement",
          "ctr_deficit": "CTR deficit vs peers", "active_day_ratio": "active-day ratio",
          "clk_m0": "clicks"}
fig, ax = plt.subplots(figsize=(7.2, 4.2))
top = imp.head(8).iloc[::-1]
ax.barh([PRETTY[f] for f in top.feature], top.AUC_drop_when_shuffled, xerr=top["std"],
        color=INK, height=0.6, error_kw={"ecolor": MUTED, "lw": 1})
ax.set_xlabel("drop in AUC when the feature is shuffled")
ax.set_title("What the model leans on", loc="left", fontsize=11)
fig.tight_layout(); fig.savefig("docs/img/importance.svg"); plt.close(fig)

drift = frame.groupby("cutoff").agg(base_rate=("y", "mean")).reset_index()
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(drift.cutoff, drift.base_rate, "o-", color=ACCENT, lw=2)
for _, r in drift.iterrows():
    ax.annotate(f"{r.base_rate:.2f}", (r.cutoff, r.base_rate), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=9)
ax.axvspan(2.5, 4.5, color=MUTED, alpha=0.15)
ax.text(3.5, 0.05, "held out in time", ha="center", fontsize=9, color="#475569")
ax.set_ylabel("share of pages that lost demand")
ax.set_xlabel("cutoff month (features up to here, label the month after)")
ax.set_title("The thing being predicted changes as the panel changes", loc="left", fontsize=11)
ax.set_ylim(0, 0.8)
fig.tight_layout(); fig.savefig("docs/img/base_rate_drift.svg"); plt.close(fig)

receipts = {"seed": SEED, "sealed_test_cutoff": CUTOFF_TEST, "sealed_label_month": "2026-06",
            "sealed_base_rate": round(float(BASE), 4), "sealed_n": int(len(test)),
            "grouped_cv": grouped.to_dict("records"), "sealed_test": sealed.to_dict("records"),
            "leak_probe": {"leaked_auc": round(float(auc_leaked), 4),
                           "honest_auc": round(float(auc_honest), 4)},
            "split_gap": {"random_auc": round(float(auc_random), 4),
                          "grouped_auc": round(float(auc_grouped), 4)},
            "permutation_importance": imp.to_dict("records")}
with open("work/outputs/capstone_metrics.json", "w") as fh:
    json.dump(receipts, fh, indent=2, default=float)
print("charts -> docs/img/ ; receipts -> work/outputs/capstone_metrics.json")

charts -> docs/img/ ; receipts -> work/outputs/capstone_metrics.json


## 8. Telling it (ML-12)

### Five-minute demo outline

1. **0:00 - The Monday problem.** An editor, six hours, 130,000 pages. Two thirds are declining. Show that a yes/no flag is useless here, so the product is an order.
2. **1:00 - The label.** Two windows on a timeline, features strictly before, outcome strictly after. Mention the per-day normalisation and why February forced it.
3. **2:00 - The rule.** Two lines of arithmetic on screen. Explain why demand is damped by a log rather than multiplied, using the signal check that showed big pages are the *stable* ones.
4. **3:00 - The result.** The precision@K chart, then the impression-recall chart. Land the sentence: same accuracy, 47x less protected traffic, because the model filled the queue with 186-impression pages.
5. **4:00 - The call.** Ship the rule. Change the objective before training again. Close on the leak probe going to 0.9997 as proof the harness was honest.

### The social cut

> Spent 8 weeks building a model to predict which web pages lose search traffic next month.
>
> On a sealed future month it tied a two-line rule on accuracy, and lost 47x on the metric that
> matters: how much at-risk traffic the queue actually protects. The model filled the list with
> pages getting 186 impressions a month. The rule picked pages getting 28,658.
>
> Shipping the rule. The model's real contribution was showing me my metric was wrong.
> Full write-up and notebooks in the comments.

### Three sentences for an employer

> I built a page-level ranking system on 1.3 million page-months of real, pseudonymised search
> performance data, predicting which content would lose search demand in the following month and
> turning the output into a ranked review queue with reason codes.
> I validated it with grouped cross-validation by client, a time-ordered split, and a sealed final
> month scored once, plus leakage probes that drove AUC to 0.9997 when I deliberately fed the model
> the answer.
> The gradient-boosted model did not beat my transparent baseline on unseen future data and was 47x
> worse at protecting at-risk traffic, so I recommended shipping the rule and rewriting the
> objective, and wrote that up as the finding rather than burying it.

## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`
- [x] My deployed paper has all 9 sections, including the Abstract at the top and Acknowledgments and data credit (the https://flyrank.ai link) at the bottom
- [x] ML-12 done in this notebook's closing cells: 5-minute demo outline, a social-post cut, and a 3-sentence employer-facing summary